In [1]:
import torch
import torch.nn as nn

def conv3x3(in_planes, out_planes, stride=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=1, bias=False)

def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride,
                     bias=False)

class BasicBlock(nn.Module):
    """
    ResNet-18, ResNet-34에서 사용하는 기본 블록.
    구조: 3x3 conv -> BN -> ReLU -> 3x3 conv -> BN -> Addition -> ReLU
    """
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(BasicBlock, self).__init__()
        # 첫 번째 3x3 합성곱
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = nn.BatchNorm2d(planes) # [cite: 285] BN은 conv 직후에 적용
        self.relu = nn.ReLU(inplace=True)
        # 두 번째 3x3 합성곱
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # 차원이 맞지 않을 경우(stride=2 등) Projection Shortcut(Option B) 적용 [cite: 279]
        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity # F(x) + x [cite: 114]
        out = self.relu(out) # [cite: 122] Addition 후 ReLU 적용

        return out

class Bottleneck(nn.Module):
    """
    ResNet-50, 101, 152에서 사용하는 병목(Bottleneck) 블록[cite: 486].
    구조: 1x1 -> 3x3 -> 1x1 (채널 확장)
    """
    expansion = 4 # 마지막 1x1 conv에서 채널이 4배로 확장됨

    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super(Bottleneck, self).__init__()
        # 1x1 conv: 차원 축소 [cite: 487]
        self.conv1 = conv1x1(inplanes, planes)
        self.bn1 = nn.BatchNorm2d(planes)
        # 3x3 conv: 병목 구간
        self.conv2 = conv3x3(planes, planes, stride)
        self.bn2 = nn.BatchNorm2d(planes)
        # 1x1 conv: 차원 복원 및 확장 (x4) [cite: 487]
        self.conv3 = conv1x1(planes, planes * self.expansion)
        self.bn3 = nn.BatchNorm2d(planes * self.expansion)
        self.relu = nn.ReLU(inplace=True)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    def __init__(self, block, layers, num_classes=1000):
        super(ResNet, self).__init__()
        self.inplanes = 64

        # 초기 입력 레이어: 7x7 conv, stride 2 [cite: 312]
        self.conv1 = nn.Conv2d(3, self.inplanes, kernel_size=7, stride=2, padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(self.inplanes)
        self.relu = nn.ReLU(inplace=True)
        # 3x3 max pooling, stride 2 [cite: 313]
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        # 각 스테이지 구성 (conv2_x ~ conv5_x)
        self.layer1 = self._make_layer(block, 64, layers[0])
        self.layer2 = self._make_layer(block, 128, layers[1], stride=2)
        self.layer3 = self._make_layer(block, 256, layers[2], stride=2)
        self.layer4 = self._make_layer(block, 512, layers[3], stride=2)

        # 최종 분류기: Average Pool + FC Layer [cite: 391]
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, num_classes)

        # 가중치 초기화 (논문에서는 [cite: 286] 방식을 언급, 여기선 PyTorch 기본값 사용)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        # Stride가 1이 아니거나 입력 채널과 출력 채널이 다를 경우 Projection Shortcut(Option B) 생성 [cite: 128, 498]
        if stride != 1 or self.inplanes != planes * block.expansion:
            downsample = nn.Sequential(
                conv1x1(self.inplanes, planes * block.expansion, stride),
                nn.BatchNorm2d(planes * block.expansion),
            )

        layers = []
        # 첫 번째 블록 (Downsample 및 Stride 적용 가능)
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes * block.expansion

        # 나머지 블록들 (Identity Shortcut)
        for _ in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)

    def forward(self, x):
        # [cite: 310-313] conv1 -> bn -> relu -> maxpool
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # [cite: 304] conv2_x ~ conv5_x
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # [cite: 391] avgpool -> fc
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x

# 논문 Table 1에 따른 모델 생성 함수들 [cite: 304]

def resnet18():
    """ResNet-18 model (BasicBlock, [2, 2, 2, 2])"""
    return ResNet(BasicBlock, [2, 2, 2, 2])

def resnet34():
    """ResNet-34 model (BasicBlock, [3, 4, 6, 3])"""
    return ResNet(BasicBlock, [3, 4, 6, 3])

def resnet50():
    """ResNet-50 model (Bottleneck, [3, 4, 6, 3])"""
    return ResNet(Bottleneck, [3, 4, 6, 3])

def resnet101():
    """ResNet-101 model (Bottleneck, [3, 4, 23, 3])"""
    return ResNet(Bottleneck, [3, 4, 23, 3])

def resnet152():
    """ResNet-152 model (Bottleneck, [3, 8, 36, 3])"""
    return ResNet(Bottleneck, [3, 8, 36, 3])

if __name__ == "__main__":
    model = resnet50()

    dummy_input = torch.randn(1, 3, 224, 224)

    # Forward Pass
    output = model(dummy_input)

    print(f"Model Architecture: ResNet-50")
    print(f"Input shape: {dummy_input.shape}")
    print(f"Output shape: {output.shape}") # 예상: torch.Size([1, 1000])

Model Architecture: ResNet-50
Input shape: torch.Size([1, 3, 224, 224])
Output shape: torch.Size([1, 1000])
